# Simulation vs Data Comparison

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/trywizard=/path/to/trywizard"` at the top of any cell.

This notebook loads real match event data, fits MLE Poisson rates, runs multiple forward simulations with those rates, and compares the simulated distributions against the observed match.

In [ ]:
!*go mod edit -replace "github.com/umbralcalc/trywizard=/Users/roberth/Code/trywizard"

In [ ]:
import (
	"fmt"
	"math"

	"github.com/umbralcalc/trywizard/pkg/match"
	"github.com/go-echarts/go-echarts/v2/charts"
	"github.com/go-echarts/go-echarts/v2/opts"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// --- Load data and compute rates ---

storage, err := match.TransformEventsToStateTimeStorage(
	"/Users/roberth/Code/trywizard/dat/events.csv", 600009, 25900,
)
if err != nil {
	panic(err)
}

rates := match.ComputeMLERates(storage)
scoreCoeffs, cardCoeffs := match.ComputeLogCoefficients(rates)

labels := []string{"home_try", "away_try", "home_conv", "away_conv", "home_yellow", "away_yellow"}
fmt.Println("MLE rates (events per minute):")
for i, r := range rates {
	fmt.Printf("  %-15s %.6f\n", labels[i], r)
}

// --- Run simulations ---

const nSims = 100
const nSteps = 80
simResults := match.RunMatchSimulations(scoreCoeffs, cardCoeffs, nSims, nSteps, 1000)
fmt.Printf("Ran %d simulations.\n", len(simResults))

// --- Real match data ---

realTrajectory := match.ComputeScoreTrajectory(storage)
realEventTotals := match.ComputeEventTotals(storage)
final := realTrajectory[len(realTrajectory)-1]
fmt.Printf("Real final score: Home %.0f - Away %.0f\n", final.Home, final.Away)

// --- Chart 1: Score trajectories ---

n := len(simResults[0].ScoreTrajectory)
meanH := make([]float64, n)
meanA := make([]float64, n)
stdH := make([]float64, n)
stdA := make([]float64, n)
for t := 0; t < n; t++ {
	var sH, sA, sH2, sA2 float64
	for _, r := range simResults {
		if t < len(r.ScoreTrajectory) {
			sH += r.ScoreTrajectory[t].Home
			sA += r.ScoreTrajectory[t].Away
			sH2 += r.ScoreTrajectory[t].Home * r.ScoreTrajectory[t].Home
			sA2 += r.ScoreTrajectory[t].Away * r.ScoreTrajectory[t].Away
		}
	}
	c := float64(nSims)
	meanH[t] = sH / c
	meanA[t] = sA / c
	stdH[t] = math.Sqrt(math.Max(sH2/c-meanH[t]*meanH[t], 0))
	stdA[t] = math.Sqrt(math.Max(sA2/c-meanA[t]*meanA[t], 0))
}

toLD := func(v []float64) []opts.LineData {
	d := make([]opts.LineData, len(v))
	for i, x := range v {
		d[i] = opts.LineData{Value: x}
	}
	return d
}

rH := make([]float64, n)
rA := make([]float64, n)
for i := 0; i < n && i < len(realTrajectory); i++ {
	rH[i] = realTrajectory[i].Home
	rA[i] = realTrajectory[i].Away
}
if len(realTrajectory) > 0 && len(realTrajectory) < n {
	last := realTrajectory[len(realTrajectory)-1]
	for i := len(realTrajectory); i < n; i++ {
		rH[i] = last.Home
		rA[i] = last.Away
	}
}

upH := make([]float64, n)
loH := make([]float64, n)
upA := make([]float64, n)
loA := make([]float64, n)
for i := 0; i < n; i++ {
	upH[i] = meanH[i] + stdH[i]
	loH[i] = math.Max(meanH[i]-stdH[i], 0)
	upA[i] = meanA[i] + stdA[i]
	loA[i] = math.Max(meanA[i]-stdA[i], 0)
}

xLabels := make([]string, n)
for i := range xLabels {
	xLabels[i] = fmt.Sprintf("%d", i)
}

line := charts.NewLine()
line.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "Score Trajectories: Real vs Simulated"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Minute"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "axis"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true)}),
)
line.SetXAxis(xLabels)
line.AddSeries("Real Home", toLD(rH), charts.WithLineStyleOpts(opts.LineStyle{Width: 3}))
line.AddSeries("Real Away", toLD(rA), charts.WithLineStyleOpts(opts.LineStyle{Width: 3}))
line.AddSeries("Sim Home (mean)", toLD(meanH), charts.WithLineStyleOpts(opts.LineStyle{Type: "dashed"}))
line.AddSeries("Sim Home +1std", toLD(upH), charts.WithLineStyleOpts(opts.LineStyle{Type: "dotted"}))
line.AddSeries("Sim Home -1std", toLD(loH), charts.WithLineStyleOpts(opts.LineStyle{Type: "dotted"}))
line.AddSeries("Sim Away (mean)", toLD(meanA), charts.WithLineStyleOpts(opts.LineStyle{Type: "dashed"}))
line.AddSeries("Sim Away +1std", toLD(upA), charts.WithLineStyleOpts(opts.LineStyle{Type: "dotted"}))
line.AddSeries("Sim Away -1std", toLD(loA), charts.WithLineStyleOpts(opts.LineStyle{Type: "dotted"}))
gonb_echarts.Display(line, "width: 1024px; height: 500px; background: white;")

// --- Chart 2: Final score scatter ---

scatter := charts.NewScatter()
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "Simulated Final Scores vs Real Outcome"}),
	charts.WithXAxisOpts(opts.XAxis{Name: "Home Score"}),
	charts.WithYAxisOpts(opts.YAxis{Name: "Away Score"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "item", Formatter: "({c})"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true)}),
)
simPts := make([]opts.ScatterData, len(simResults))
for i, r := range simResults {
	if len(r.ScoreTrajectory) > 0 {
		last := r.ScoreTrajectory[len(r.ScoreTrajectory)-1]
		simPts[i] = opts.ScatterData{Value: []interface{}{last.Home, last.Away}, SymbolSize: 8}
	}
}
scatter.AddSeries("Simulated", simPts)
scatter.AddSeries("Real", []opts.ScatterData{{
	Value: []interface{}{final.Home, final.Away}, SymbolSize: 20,
}})
gonb_echarts.Display(scatter, "width: 800px; height: 500px; background: white;")

// --- Chart 3: Event totals bar chart ---

meanEv := make([]float64, match.EventWidth)
stdEv := make([]float64, match.EventWidth)
for j := 0; j < match.EventWidth; j++ {
	var s, s2 float64
	for _, r := range simResults {
		v := r.EventTotals[j]
		s += v
		s2 += v * v
	}
	c := float64(len(simResults))
	meanEv[j] = s / c
	stdEv[j] = math.Sqrt(math.Max(s2/c-meanEv[j]*meanEv[j], 0))
}

fmt.Printf("\n%-15s %8s %8s %8s\n", "Event Type", "Real", "Sim Mean", "Sim Std")
fmt.Println("----------------------------------------------")
for i, label := range labels {
	fmt.Printf("%-15s %8.0f %8.2f %8.2f\n", label, realEventTotals[i], meanEv[i], stdEv[i])
}

bar := charts.NewBar()
bar.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{Title: "Event Totals: Real vs Simulated Mean"}),
	charts.WithTooltipOpts(opts.Tooltip{Trigger: "axis"}),
	charts.WithLegendOpts(opts.Legend{Show: opts.Bool(true)}),
)
bar.SetXAxis(labels)
realBars := make([]opts.BarData, match.EventWidth)
simBars := make([]opts.BarData, match.EventWidth)
for i := 0; i < match.EventWidth; i++ {
	realBars[i] = opts.BarData{Value: realEventTotals[i]}
	simBars[i] = opts.BarData{Value: math.Round(meanEv[i]*100) / 100}
}
bar.AddSeries("Real", realBars)
bar.AddSeries("Simulated (mean)", simBars)
gonb_echarts.Display(bar, "width: 800px; height: 400px; background: white;")